# ML Analysis with PredefinedSplit Validation — **raw (continuous) age**

This notebook is `ml_analysis_predefined_val.ipynb` with the age feature changed from a
3-level bin to the raw `anchor_age` value, plus two related fixes.

**Why the change.** With `age_mapped` (3 bins) x `race_mapped` (3) x `gender` (2) the
feature space held only **18 distinct rows**. Every model that is flexible enough
collapses onto the same object — the table of `P(public | cell)` — so the five
"models" were five implementations of one contingency table (the Decision Tree
reproduced it exactly), the hyperparameter search was vacuous, and the AUC was capped
at **0.6544**, the best any function of those three features could achieve.

**What changed here**

| | old notebook | this notebook |
|---|---|---|
| age | 3 bins (`young`/`adult`/`senior`) | raw `anchor_age`, continuous (18-64) |
| race | `LabelEncoder` -> 0/1/2 | one-hot (nominal, no false ordering) |
| gender | `LabelEncoder` -> 0/1 | unchanged (binary, so 0/1 is fine) |
| KNN scaling | none | `StandardScaler` inside a `Pipeline` |
| cohort | dropped age >= 64 as "elderly" | **keeps every patient** |
| KNN importances | `best_knn.feature_importances_` (raises) | permutation importance |

**Cohort note.** The old age bins discarded `elderly`, which in this data meant exactly
the 36 images at age 64 (`anchor_age` is already capped at 18-64). Raw age gives no
reason to drop them, so the test set here is the full **1965 images / 1224 patients** —
which is exactly the test set the CXR models in `bootstrap_results/exp0` were evaluated
on, making the two directly comparable image-for-image.

Everything else — `PredefinedSplit` tuning on the real validation set, retraining on
`X_train` only, patient-level bootstrap CIs — is unchanged.

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import PredefinedSplit, GridSearchCV
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score)


# ── Preprocessing ─────────────────────────────────────────────────────────────
#
# Differences from ml_analysis_predefined_val.ipynb are marked CHANGED.

def map_insurance_type(insurance):
    if insurance in ["Medicaid", "Medicare"]:
        return "public"
    elif insurance == "Private":
        return "private"
    return "others"

def map_race(race: str) -> str:
    if race == 'WHITE':
        return 'white'
    elif race == 'BLACK':
        return 'black'
    return 'others'

# CHANGED: no map_age(). anchor_age is used as-is.

def age_band(age):
    """Reporting only — never a model feature. Used for subgroup breakdowns."""
    if age < 40:
        return 'Young'
    elif age < 50:
        return 'Adult'
    return 'Senior'


PATIENT_ID_CANDIDATES = ['subject_id_x', 'subject_id', 'subject_idx']

def find_patient_col(df):
    for name in PATIENT_ID_CANDIDATES:
        if name in df.columns:
            return name
    return None


def preprocess_raw(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df['insurance_mapped'] = df['new_insurance_type'].apply(map_insurance_type)
    df['race_mapped']      = df['race'].apply(map_race)
    df['age']              = df['anchor_age'].astype(float)   # CHANGED: raw age
    df['age_band']         = df['anchor_age'].apply(age_band)  # reporting only

    pid_col = find_patient_col(df)
    if pid_col is not None:
        df['patient_id'] = df[pid_col].astype(str)
    if 'dicom_id' in df.columns:
        df['dicom_id'] = df['dicom_id'].astype(str)

    df = df[df['insurance_mapped'].isin(['public', 'private'])]
    # CHANGED: no age filter — the old 'elderly' bin dropped the age-64 patients.
    keep = ['age', 'age_band', 'race_mapped', 'gender', 'insurance_mapped',
            'patient_id', 'dicom_id']
    return df[[c for c in df.columns if c in keep]]


# Fixed column order so train/val/test always encode identically
RACE_LEVELS = ['black', 'others', 'white']
FEATURES = ['age', 'gender'] + [f'race_{r}' for r in RACE_LEVELS]

def encode_features(df: pd.DataFrame):
    """CHANGED: continuous age + one-hot race. No ordinal codes for nominal vars."""
    X = pd.DataFrame(index=df.index)
    X['age']    = df['age'].astype(float)
    X['gender'] = (df['gender'] == 'M').astype(int)      # F=0, M=1
    for r in RACE_LEVELS:
        X[f'race_{r}'] = (df['race_mapped'] == r).astype(int)
    y = (df['insurance_mapped'] == 'public').astype(int)  # public = 1
    return X[FEATURES], y


def prepare_split(df):
    proc = preprocess_raw(df).reset_index(drop=True)
    X, y = encode_features(proc)
    return (X.reset_index(drop=True), y.reset_index(drop=True), proc)


# ── Load ──────────────────────────────────────────────────────────────────────
df_train = pd.read_csv("./insurance_dataset_8_1_1_PMMthree_train_medgemmaChecked.csv")
df_val   = pd.read_csv("./insurance_dataset_8_1_1_PMMthree_val_medgemmaChecked.csv")
df_test  = pd.read_csv("./insurance_dataset_8_1_1_PMMthree_test_medgemmaChecked.csv")

X_train, y_train, meta_train = prepare_split(df_train)
X_val,   y_val,   meta_val   = prepare_split(df_val)
X_test,  y_test,  meta_test  = prepare_split(df_test)

patients_train, patients_val, patients_test = (meta_train['patient_id'],
                                               meta_val['patient_id'],
                                               meta_test['patient_id'])

print(f"Train: {len(X_train)}, Val: {len(X_val)}, Test: {len(X_test)}")
print(f"Unique patients — Train: {patients_train.nunique()}, "
      f"Val: {patients_val.nunique()}, Test: {patients_test.nunique()}")
print(f"\nFeatures ({len(FEATURES)}): {FEATURES}")
print(f"Age range: {X_train['age'].min():.0f}-{X_train['age'].max():.0f}, "
      f"{X_train['age'].nunique()} distinct values in train")
print(f"Positive class = public insurance; train prevalence "
      f"{y_train.mean():.3f}, test {y_test.mean():.3f}")

# Patient-disjointness is what actually matters for validity here
overlap = (set(patients_train) & set(patients_test),
           set(patients_train) & set(patients_val),
           set(patients_val)   & set(patients_test))
print(f"\nPatient overlap train/test {len(overlap[0])}, train/val {len(overlap[1])}, "
      f"val/test {len(overlap[2])}  (must all be 0)")

Train: 15494, Val: 1905, Test: 1965
Unique patients — Train: 9512, Val: 1199, Test: 1224

Features (5): ['age', 'gender', 'race_black', 'race_others', 'race_white']
Age range: 18-64, 47 distinct values in train
Positive class = public insurance; train prevalence 0.562, test 0.585

Patient overlap train/test 0, train/val 0, val/test 0  (must all be 0)


In [2]:
# ── Build PredefinedSplit ─────────────────────────────────────────────────────
#
# test_fold marks each sample: -1 → always train, 0 → always validate.
# GridSearchCV then sees exactly one "fold": fit on X_train, score on X_val.

X_trainval = pd.concat([X_train, X_val], ignore_index=True)
y_trainval = pd.concat([y_train, y_val], ignore_index=True)

split_index = np.array([-1] * len(X_train) + [0] * len(X_val))
ps = PredefinedSplit(test_fold=split_index)

print(f"Combined trainval size : {len(X_trainval)}")
print(f"Training fold size     : {(split_index == -1).sum()}")
print(f"Validation fold size   : {(split_index ==  0).sum()}")

Combined trainval size : 17399
Training fold size     : 15494
Validation fold size   : 1905


In [3]:
# ── Shared evaluation helper ──────────────────────────────────────────────────

def evaluate_model(y_true, y_pred, y_prob, model_name, split_name):
    print(f"\n{model_name} — {split_name}")
    print(f"  Accuracy : {accuracy_score(y_true, y_pred):.4f}")
    print(f"  Precision: {precision_score(y_true, y_pred, zero_division=0):.4f}")
    print(f"  Recall   : {recall_score(y_true, y_pred, zero_division=0):.4f}")
    print(f"  F1       : {f1_score(y_true, y_pred, zero_division=0):.4f}")
    try:
        print(f"  AUC      : {roc_auc_score(y_true, y_prob):.4f}")
    except Exception:
        print("  AUC      : N/A")


def show_importances(values, features, title):
    imp = (pd.DataFrame({'Feature': features, 'Importance': values})
           .sort_values('Importance', ascending=False).reset_index(drop=True))
    print(f"\n{title}")
    print(imp.to_string(index=False))
    return imp

## Random Forest

In [4]:
from sklearn.ensemble import RandomForestClassifier

param_grid_rf = {
    'n_estimators':      [100, 200, 300],
    'max_depth':         [5, 10, 20, None],
    'min_samples_split': [2, 5],
    'min_samples_leaf':  [1, 2, 10],
    'class_weight':      [None, 'balanced'],
}

grid_search_rf = GridSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_grid=param_grid_rf, cv=ps, scoring='roc_auc',
    n_jobs=-1, verbose=1, refit=False,
)
grid_search_rf.fit(X_trainval, y_trainval)

print("\nBest params:", grid_search_rf.best_params_)
print(f"Best val AUC: {grid_search_rf.best_score_:.4f}")

best_rf = RandomForestClassifier(**grid_search_rf.best_params_, random_state=42)
best_rf.fit(X_train, y_train)

evaluate_model(y_val,  best_rf.predict(X_val),  best_rf.predict_proba(X_val)[:,1],  "Random Forest", "Validation")
evaluate_model(y_test, best_rf.predict(X_test), best_rf.predict_proba(X_test)[:,1], "Random Forest", "Test")

Fitting 1 folds for each of 144 candidates, totalling 144 fits

Best params: {'class_weight': 'balanced', 'max_depth': 5, 'min_samples_leaf': 10, 'min_samples_split': 2, 'n_estimators': 100}
Best val AUC: 0.5768

Random Forest — Validation
  Accuracy : 0.5323
  Precision: 0.5965
  Recall   : 0.4300
  F1       : 0.4997
  AUC      : 0.5768

Random Forest — Test
  Accuracy : 0.5786
  Precision: 0.6931
  Recall   : 0.5013
  F1       : 0.5818
  AUC      : 0.6279


In [5]:
_ = show_importances(best_rf.feature_importances_, X_train.columns,
                     "Random Forest — feature importances")


Random Forest — feature importances
    Feature  Importance
        age    0.404712
 race_white    0.303447
 race_black    0.203328
race_others    0.053131
     gender    0.035382


## XGBoost

In [6]:
import xgboost as xgb

scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

param_grid_xgb = {
    'n_estimators':     [100, 200],
    'max_depth':        [2, 3, 5],
    'learning_rate':    [0.01, 0.1],
    'subsample':        [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0],
    'min_child_weight': [1, 10],
    'scale_pos_weight': [scale_pos_weight],
}

grid_search_xgb = GridSearchCV(
    estimator=xgb.XGBClassifier(random_state=42, eval_metric='logloss', verbosity=0),
    param_grid=param_grid_xgb, cv=ps, scoring='roc_auc',
    n_jobs=-1, verbose=1, refit=False,
)
grid_search_xgb.fit(X_trainval, y_trainval)

print("\nBest params:", grid_search_xgb.best_params_)
print(f"Best val AUC: {grid_search_xgb.best_score_:.4f}")

best_xgb = xgb.XGBClassifier(**grid_search_xgb.best_params_, random_state=42,
                             eval_metric='logloss', verbosity=0)
best_xgb.fit(X_train, y_train)

evaluate_model(y_val,  best_xgb.predict(X_val),  best_xgb.predict_proba(X_val)[:,1],  "XGBoost", "Validation")
evaluate_model(y_test, best_xgb.predict(X_test), best_xgb.predict_proba(X_test)[:,1], "XGBoost", "Test")

Fitting 1 folds for each of 96 candidates, totalling 96 fits

Best params: {'colsample_bytree': 0.8, 'learning_rate': 0.1, 'max_depth': 2, 'min_child_weight': 10, 'n_estimators': 100, 'scale_pos_weight': 0.7798966111430212, 'subsample': 0.8}
Best val AUC: 0.5795

XGBoost — Validation
  Accuracy : 0.5459
  Precision: 0.6062
  Recall   : 0.4686
  F1       : 0.5286
  AUC      : 0.5795

XGBoost — Test
  Accuracy : 0.5832
  Precision: 0.6897
  Recall   : 0.5222
  F1       : 0.5944
  AUC      : 0.6423


In [7]:
_ = show_importances(best_xgb.feature_importances_, X_train.columns,
                     "XGBoost — feature importances")


XGBoost — feature importances
    Feature  Importance
 race_white    0.687994
 race_black    0.143711
        age    0.111546
     gender    0.031655
race_others    0.025094


## CatBoost

In [8]:
from catboost import CatBoostClassifier

param_grid_catboost = {
    'iterations':          [300, 500],
    'learning_rate':       [0.03, 0.05, 0.1],
    'depth':               [4, 6, 8],
    'l2_leaf_reg':         [5, 10],
    'auto_class_weights':  ['Balanced'],
    'bagging_temperature': [0.5, 1.0],
}

grid_search_catboost = GridSearchCV(
    estimator=CatBoostClassifier(random_state=42, verbose=0, eval_metric='AUC'),
    param_grid=param_grid_catboost, cv=ps, scoring='roc_auc',
    n_jobs=-1, verbose=1, refit=False,
)
grid_search_catboost.fit(X_trainval, y_trainval)

print("\nBest params:", grid_search_catboost.best_params_)
print(f"Best val AUC: {grid_search_catboost.best_score_:.4f}")

best_catboost = CatBoostClassifier(**grid_search_catboost.best_params_,
                                   random_state=42, verbose=0, eval_metric='AUC')
best_catboost.fit(X_train, y_train)

evaluate_model(y_val,  best_catboost.predict(X_val),  best_catboost.predict_proba(X_val)[:,1],  "CatBoost", "Validation")
evaluate_model(y_test, best_catboost.predict(X_test), best_catboost.predict_proba(X_test)[:,1], "CatBoost", "Test")

Fitting 1 folds for each of 72 candidates, totalling 72 fits

Best params: {'auto_class_weights': 'Balanced', 'bagging_temperature': 0.5, 'depth': 4, 'iterations': 300, 'l2_leaf_reg': 5, 'learning_rate': 0.03}
Best val AUC: 0.5783

CatBoost — Validation
  Accuracy : 0.5438
  Precision: 0.6070
  Recall   : 0.4551
  F1       : 0.5202
  AUC      : 0.5783

CatBoost — Test
  Accuracy : 0.5761
  Precision: 0.6894
  Recall   : 0.5004
  F1       : 0.5799
  AUC      : 0.6268


In [9]:
_ = show_importances(best_catboost.feature_importances_, X_train.columns,
                     "CatBoost — feature importances")


CatBoost — feature importances
    Feature  Importance
        age   39.347922
 race_white   28.067023
 race_black   16.080645
race_others    8.740921
     gender    7.763488


## KNN

Two fixes relative to the old notebook. Raw age spans 18-64 while the one-hot race
columns are 0/1, so an unscaled Euclidean metric would be almost entirely age —
a `StandardScaler` now sits inside the `Pipeline` so the grid search scales using
training-fold statistics only. And race is one-hot rather than ordinal-coded, so
the metric no longer treats `black`->`others` as nearer than `black`->`white`.

In [10]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

pipe_knn = Pipeline([('scale', StandardScaler()),
                     ('knn', KNeighborsClassifier())])

param_grid_knn = {
    'knn__n_neighbors': [5, 15, 25, 50, 100],
    'knn__weights':     ['uniform', 'distance'],
    'knn__metric':      ['euclidean', 'manhattan'],
}

grid_search_knn = GridSearchCV(
    estimator=pipe_knn, param_grid=param_grid_knn, cv=ps,
    scoring='roc_auc', n_jobs=-1, verbose=1, refit=False,
)
grid_search_knn.fit(X_trainval, y_trainval)

print("\nBest params:", grid_search_knn.best_params_)
print(f"Best val AUC: {grid_search_knn.best_score_:.4f}")

best_knn = Pipeline([('scale', StandardScaler()), ('knn', KNeighborsClassifier())])
best_knn.set_params(**grid_search_knn.best_params_)
best_knn.fit(X_train, y_train)

evaluate_model(y_val,  best_knn.predict(X_val),  best_knn.predict_proba(X_val)[:,1],  "KNN", "Validation")
evaluate_model(y_test, best_knn.predict(X_test), best_knn.predict_proba(X_test)[:,1], "KNN", "Test")

Fitting 1 folds for each of 20 candidates, totalling 20 fits

Best params: {'knn__metric': 'manhattan', 'knn__n_neighbors': 50, 'knn__weights': 'uniform'}
Best val AUC: 0.5649

KNN — Validation
  Accuracy : 0.5417
  Precision: 0.5623
  Recall   : 0.7063
  F1       : 0.6261
  AUC      : 0.5649

KNN — Test
  Accuracy : 0.5842
  Precision: 0.6213
  Recall   : 0.7398
  F1       : 0.6754
  AUC      : 0.5762


In [11]:
# KNN exposes no feature_importances_ (the old notebook's cell raised
# AttributeError here). Permutation importance on the VALIDATION set is the
# model-agnostic equivalent; the test set stays untouched.
from sklearn.inspection import permutation_importance

perm = permutation_importance(best_knn, X_val, y_val, scoring='roc_auc',
                              n_repeats=20, random_state=42, n_jobs=-1)
_ = show_importances(perm.importances_mean, X_val.columns,
                     "KNN — permutation importance (drop in val AUC when shuffled)")


KNN — permutation importance (drop in val AUC when shuffled)
    Feature  Importance
        age    0.034793
 race_black    0.032212
race_others    0.017635
     gender    0.010008
 race_white    0.006055


## Decision Tree

In [12]:
from sklearn.tree import DecisionTreeClassifier

param_grid_dt = {
    'max_depth':         [3, 5, 10, 20, None],
    'min_samples_split': [2, 5, 20],
    'min_samples_leaf':  [1, 2, 10, 50],
    'criterion':         ['gini', 'entropy'],
    'class_weight':      [None, 'balanced'],
}

grid_search_dt = GridSearchCV(
    estimator=DecisionTreeClassifier(random_state=42),
    param_grid=param_grid_dt, cv=ps, scoring='roc_auc',
    n_jobs=-1, verbose=1, refit=False,
)
grid_search_dt.fit(X_trainval, y_trainval)

print("\nBest params:", grid_search_dt.best_params_)
print(f"Best val AUC: {grid_search_dt.best_score_:.4f}")

best_dt = DecisionTreeClassifier(**grid_search_dt.best_params_, random_state=42)
best_dt.fit(X_train, y_train)

evaluate_model(y_val,  best_dt.predict(X_val),  best_dt.predict_proba(X_val)[:,1],  "Decision Tree", "Validation")
evaluate_model(y_test, best_dt.predict(X_test), best_dt.predict_proba(X_test)[:,1], "Decision Tree", "Test")

Fitting 1 folds for each of 240 candidates, totalling 240 fits

Best params: {'class_weight': 'balanced', 'criterion': 'gini', 'max_depth': 5, 'min_samples_leaf': 1, 'min_samples_split': 2}
Best val AUC: 0.5638

Decision Tree — Validation
  Accuracy : 0.5333
  Precision: 0.5989
  Recall   : 0.4271
  F1       : 0.4986
  AUC      : 0.5638

Decision Tree — Test
  Accuracy : 0.5817
  Precision: 0.7031
  Recall   : 0.4926
  F1       : 0.5793
  AUC      : 0.6286


In [13]:
_ = show_importances(best_dt.feature_importances_, X_train.columns,
                     "Decision Tree — feature importances")


Decision Tree — feature importances
    Feature  Importance
 race_white    0.537802
        age    0.375364
 race_black    0.054140
     gender    0.032694
race_others    0.000000


## Model Comparison

In [14]:
models = {
    "Random Forest" : best_rf,
    "XGBoost"       : best_xgb,
    "CatBoost"      : best_catboost,
    "KNN"           : best_knn,
    "Decision Tree" : best_dt,
}

rows = []
for name, model in models.items():
    for split_name, X, y in [("Val", X_val, y_val), ("Test", X_test, y_test)]:
        y_pred = model.predict(X)
        y_prob = model.predict_proba(X)[:, 1]
        rows.append({
            "Model": name, "Split": split_name,
            "AUC": roc_auc_score(y, y_prob),
            "Accuracy": accuracy_score(y, y_pred),
            "F1": f1_score(y, y_pred, zero_division=0),
        })

comparison_df = pd.DataFrame(rows).set_index(["Model", "Split"]).round(4)
print(comparison_df.to_string())

                        AUC  Accuracy      F1
Model         Split                          
Random Forest Val    0.5768    0.5323  0.4997
              Test   0.6279    0.5786  0.5818
XGBoost       Val    0.5795    0.5459  0.5286
              Test   0.6423    0.5832  0.5944
CatBoost      Val    0.5783    0.5438  0.5202
              Test   0.6268    0.5761  0.5799
KNN           Val    0.5649    0.5417  0.6261
              Test   0.5762    0.5842  0.6754
Decision Tree Val    0.5638    0.5333  0.4986
              Test   0.6286    0.5817  0.5793


## Feature-space diagnostics

The point of the change. The old notebook's three binned features spanned only 18
distinct rows, which capped the AUC and made every model collapse onto the same
contingency table. This cell measures whether raw age actually escaped that.

In [15]:
# How many distinct feature rows / distinct predicted scores now exist?
cells_old = meta_test.assign(b=meta_test['age_band']).groupby(
    ['b', 'race_mapped', 'gender']).ngroups
cells_new = X_test.drop_duplicates().shape[0]
print(f"Distinct feature rows in the test set")
print(f"  binned age (old encoding) : {cells_old}")
print(f"  raw age    (this notebook): {cells_new}")

print(f"\nDistinct predicted probabilities on the test set (was 8-18 in the old notebook):")
for name, model in models.items():
    p = model.predict_proba(X_test)[:, 1]
    print(f"  {name:<15} {len(np.unique(p)):>5} distinct   "
          f"largest tie block {np.unique(p, return_counts=True)[1].max():>5}")

# The old ceiling: best AUC achievable by ANY function of (age_band, race, gender),
# computed by scoring each test image with its own cell's observed public rate.
# Fitted on the test set itself, so it is an optimistic upper bound.
cell_key = list(zip(meta_test['age_band'], meta_test['race_mapped'], meta_test['gender']))
rate = pd.Series(y_test.values, index=pd.MultiIndex.from_tuples(cell_key)).groupby(
    level=[0, 1, 2]).mean()
oracle_binned = np.array([rate[k] for k in cell_key])
auc_ceiling_old = roc_auc_score(y_test, oracle_binned)

print(f"\nCeiling for binned demographics (optimistic, fitted on test): "
      f"{auc_ceiling_old:.4f}")
print(f"Raw-age model AUCs on the same test set:")
for name, model in models.items():
    a = roc_auc_score(y_test, model.predict_proba(X_test)[:, 1])
    flag = "  ABOVE the old ceiling" if a > auc_ceiling_old else ""
    print(f"  {name:<15} {a:.4f}{flag}")

Distinct feature rows in the test set
  binned age (old encoding) : 18
  raw age    (this notebook): 263

Distinct predicted probabilities on the test set (was 8-18 in the old notebook):
  Random Forest     230 distinct   largest tie block    42
  XGBoost           204 distinct   largest tie block    42
  CatBoost          263 distinct   largest tie block    40
  KNN                37 distinct   largest tie block   197
  Decision Tree      27 distinct   largest tie block   444

Ceiling for binned demographics (optimistic, fitted on test): 0.6546
Raw-age model AUCs on the same test set:
  Random Forest   0.6279
  XGBoost         0.6423
  CatBoost        0.6268
  KNN             0.5762
  Decision Tree   0.6286


## Bootstrap Evaluation on Test Set

Patient-level (cluster) bootstrap, identical in method to `bootstrap_evaluate.py` and to
the old notebook: whole patients are drawn with replacement and all their images come
along, because images from one patient are correlated and image-level resampling would
understate the CIs.

**Full** is the point estimate on the entire test set with no resampling — the number to
report. The bootstrap columns give the uncertainty around it.

In [16]:
from collections import defaultdict

# Subgroups are read from the reporting frame (strings), not from model features —
# the model sees raw age, the report still breaks results down by age band.
SUBGROUP_COLS = {'Gender': 'gender', 'Age': 'age_band', 'Race': 'race_mapped'}
MIN_GROUP_SIZE = 10


def _compute_metrics(y_true, y_prob, y_pred):
    try:
        auc = roc_auc_score(y_true, y_prob)
    except Exception:
        auc = float('nan')
    return {'AUC': auc,
            'Accuracy':  accuracy_score(y_true, y_pred),
            'Precision': precision_score(y_true, y_pred, zero_division=0),
            'Recall':    recall_score(y_true, y_pred, zero_division=0),
            'F1':        f1_score(y_true, y_pred, zero_division=0)}


def build_clusters(labels):
    members, order = {}, []
    for pos, lab in enumerate(labels):
        if lab not in members:
            members[lab] = []
            order.append(lab)
        members[lab].append(pos)
    return [np.array(members[l], dtype=np.int64) for l in order]


def evaluate_full_dataset(y_true, y_prob, y_pred, demo_df, patients,
                          min_group_size=MIN_GROUP_SIZE):
    y_true, y_prob, y_pred = map(np.asarray, (y_true, y_prob, y_pred))
    pat = np.asarray(patients)

    def _row(mask):
        m = _compute_metrics(y_true[mask], y_prob[mask], y_pred[mask])
        m['n_samples']  = int(mask.sum())
        m['n_patients'] = int(len(np.unique(pat[mask])))
        return m

    full = {'Overall': _row(np.ones(len(y_true), dtype=bool))}
    for gname, col in SUBGROUP_COLS.items():
        demo = np.asarray(demo_df[col])
        for val in sorted(pd.unique(demo)):
            mask = demo == val
            if mask.sum() < min_group_size:
                continue
            full[f'{gname}: {val}'] = _row(mask)
    return full


def bootstrap_evaluate_sklearn(y_true, y_prob, y_pred, demo_df, patients,
                               n_bootstrap=1000, seed=42,
                               min_group_size=MIN_GROUP_SIZE):
    y_true, y_prob, y_pred = map(np.asarray, (y_true, y_prob, y_pred))
    demo_arrays = {g: np.asarray(demo_df[c]) for g, c in SUBGROUP_COLS.items()}

    clusters = build_clusters(list(np.asarray(patients)))
    n_draw = len(clusters)
    info = {'unit': 'patient', 'n_samples': len(y_true),
            'n_patients': len(clusters), 'n_draw': n_draw}

    rng = np.random.RandomState(seed)
    all_results = []
    for i in range(n_bootstrap):
        picked = rng.choice(len(clusters), size=n_draw, replace=True)
        idx = np.concatenate([clusters[c] for c in picked])
        yt, yprob, ypred = y_true[idx], y_prob[idx], y_pred[idx]

        m = _compute_metrics(yt, yprob, ypred)
        m['group'], m['iteration'] = 'Overall', i
        all_results.append(m)

        for gname, col in SUBGROUP_COLS.items():
            demo = demo_arrays[gname][idx]
            for val in sorted(pd.unique(demo_arrays[gname])):
                mask = demo == val
                if mask.sum() < min_group_size:
                    continue
                sm = _compute_metrics(yt[mask], yprob[mask], ypred[mask])
                sm['group'], sm['iteration'] = f'{gname}: {val}', i
                all_results.append(sm)
    return all_results, info


def summarize_bootstrap(all_results, full_metrics=None):
    groups = defaultdict(lambda: defaultdict(list))
    for r in all_results:
        for metric in METRIC_NAMES:
            groups[r['group']][metric].append(r[metric])

    summary = []
    for gname, md_ in groups.items():
        row = {'group': gname}
        full = (full_metrics or {}).get(gname, {})
        row['n_samples']  = full.get('n_samples',  float('nan'))
        row['n_patients'] = full.get('n_patients', float('nan'))
        for metric, values in md_.items():
            arr = np.array(values)
            arr = arr[~np.isnan(arr)]
            row[f'{metric}_full'] = full.get(metric, float('nan'))
            if len(arr) == 0:
                for s in ('mean', 'std', 'ci_lower', 'ci_upper'):
                    row[f'{metric}_{s}'] = float('nan')
            else:
                row[f'{metric}_mean']     = np.mean(arr)
                row[f'{metric}_std']      = np.std(arr)
                row[f'{metric}_ci_lower'] = np.percentile(arr, 2.5)
                row[f'{metric}_ci_upper'] = np.percentile(arr, 97.5)
        summary.append(row)
    return summary


def print_bootstrap_summary(summary, model_name, n_bootstrap, info):
    print(f"\n{'='*92}")
    print(f"{model_name}  —  Bootstrap Results (N={n_bootstrap})")
    print(f"Resampling unit: patient ({info['n_draw']} of {info['n_patients']} patients "
          f"per iteration, {info['n_samples']} images in the test set)")
    print(f"{'='*92}")
    order = ['Overall'] + sorted(s['group'] for s in summary if s['group'] != 'Overall')
    for gname in order:
        row = next((s for s in summary if s['group'] == gname), None)
        if row is None:
            continue
        print(f"\n--- {gname} (n={row['n_samples']} images, {row['n_patients']} patients) ---")
        print(f"{'Metric':<12} | {'Full':>8} | {'Mean':>8} | {'Std':>8} | {'95% CI':>22}")
        print(f"{'-'*12}-+-{'-'*8}-+-{'-'*8}-+-{'-'*8}-+-{'-'*22}")
        for m in METRIC_NAMES:
            print(f"{m:<12} | {row[f'{m}_full']:>8.4f} | {row[f'{m}_mean']:>8.4f} | "
                  f"{row[f'{m}_std']:>8.4f} | "
                  f"[{row[f'{m}_ci_lower']:.4f}, {row[f'{m}_ci_upper']:.4f}]")

In [17]:
import os, csv

METRIC_NAMES = ['AUC', 'Precision', 'Recall', 'F1', 'Accuracy']
N_BOOTSTRAP  = 1000
SEED         = 42
OUTPUT_DIR   = "bootstrap_results_ml_rawage"

GENDER_CODE = {'F': 0, 'M': 1}
AGE_CODE    = {'Adult': 0, 'Senior': 1, 'Young': 2}
RACE_CODE   = {'black': 0, 'others': 1, 'white': 2}


def save_predictions(meta, y_true, y_prob, y_pred, output_dir, model_name):
    """Per-sample predictions in the same schema bootstrap_evaluate.py writes, so
    statistical_testing_DeLong.py can be pointed straight at these files."""
    os.makedirs(output_dir, exist_ok=True)
    path = os.path.join(output_dir, f"{model_name.replace(' ', '_')}_predictions.csv")
    fields = (['sample_id', 'patient_id', 'true_label', 'pred_label', 'correct',
               'logit_0', 'logit_1', 'prob_0', 'prob_1',
               'gender', 'age', 'race', 'gender_name', 'age_name', 'race_name'])
    with open(path, 'w', newline='') as f:
        w = csv.DictWriter(f, fieldnames=fields)
        w.writeheader()
        for i in range(len(y_true)):
            p1 = float(y_prob[i]); p0 = 1.0 - p1
            w.writerow({
                'sample_id':  meta['dicom_id'].iloc[i],
                'patient_id': meta['patient_id'].iloc[i],
                'true_label': int(y_true.iloc[i]),
                'pred_label': int(y_pred[i]),
                'correct':    int(y_pred[i] == y_true.iloc[i]),
                'logit_0': float(np.log(max(p0, 1e-12))),
                'logit_1': float(np.log(max(p1, 1e-12))),
                'prob_0': p0, 'prob_1': p1,
                'gender': GENDER_CODE[meta['gender'].iloc[i]],
                'age':    AGE_CODE[meta['age_band'].iloc[i]],
                'race':   RACE_CODE[meta['race_mapped'].iloc[i]],
                'gender_name': 'Female' if meta['gender'].iloc[i] == 'F' else 'Male',
                'age_name':    meta['age_band'].iloc[i],
                'race_name':   meta['race_mapped'].iloc[i].capitalize(),
            })
    return path


def save_bootstrap_results(all_results, summary, full_metrics, output_dir, model_name):
    os.makedirs(output_dir, exist_ok=True)
    safe = model_name.replace(' ', '_')

    with open(os.path.join(output_dir, f"{safe}_bootstrap_iterations.csv"), 'w', newline='') as f:
        fields = ['group', 'iteration'] + METRIC_NAMES
        w = csv.DictWriter(f, fieldnames=fields); w.writeheader()
        for r in all_results:
            w.writerow({k: r[k] for k in fields})

    fields = ['group', 'n_samples', 'n_patients']
    for m in METRIC_NAMES:
        fields += [f'{m}_full', f'{m}_mean', f'{m}_std', f'{m}_ci_lower', f'{m}_ci_upper']
    with open(os.path.join(output_dir, f"{safe}_bootstrap_summary.csv"), 'w', newline='') as f:
        w = csv.DictWriter(f, fieldnames=fields); w.writeheader()
        for row in summary:
            w.writerow({k: row.get(k, '') for k in fields})

    fields = ['group', 'n_samples', 'n_patients'] + METRIC_NAMES
    with open(os.path.join(output_dir, f"{safe}_full_test_metrics.csv"), 'w', newline='') as f:
        w = csv.DictWriter(f, fieldnames=fields); w.writeheader()
        for gname, m in full_metrics.items():
            row = {'group': gname}
            row.update({k: m.get(k, '') for k in fields if k != 'group'})
            w.writerow(row)


print(f"Test set   : {len(X_test)} images / {patients_test.nunique()} patients")
print(f"Iterations : {N_BOOTSTRAP}   Output dir: {OUTPUT_DIR}\n")

all_summaries, all_iter_results, all_full_metrics = {}, {}, {}

for model_name, model in models.items():
    print(f"--- {model_name} ---")
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    full_metrics = evaluate_full_dataset(y_test, y_prob, y_pred, meta_test, patients_test)
    results, info = bootstrap_evaluate_sklearn(
        y_test, y_prob, y_pred, meta_test, patients_test,
        n_bootstrap=N_BOOTSTRAP, seed=SEED)
    summary = summarize_bootstrap(results, full_metrics=full_metrics)

    all_iter_results[model_name] = results
    all_summaries[model_name]    = summary
    all_full_metrics[model_name] = full_metrics

    print_bootstrap_summary(summary, model_name, N_BOOTSTRAP, info)
    save_bootstrap_results(results, summary, full_metrics, OUTPUT_DIR, model_name)
    p = save_predictions(meta_test, y_test, y_prob, y_pred, OUTPUT_DIR, model_name)
    print(f"  Predictions saved: {p}\n")

Test set   : 1965 images / 1224 patients
Iterations : 1000   Output dir: bootstrap_results_ml_rawage

--- Random Forest ---

Random Forest  —  Bootstrap Results (N=1000)
Resampling unit: patient (1224 of 1224 patients per iteration, 1965 images in the test set)

--- Overall (n=1965 images, 1224 patients) ---
Metric       |     Full |     Mean |      Std |                 95% CI
-------------+----------+----------+----------+-----------------------
AUC          |   0.6279 |   0.6277 |   0.0183 | [0.5920, 0.6641]
Precision    |   0.6931 |   0.6919 |   0.0244 | [0.6430, 0.7372]
Recall       |   0.5013 |   0.5004 |   0.0274 | [0.4509, 0.5556]
F1           |   0.5818 |   0.5804 |   0.0240 | [0.5327, 0.6291]
Accuracy     |   0.5786 |   0.5785 |   0.0182 | [0.5433, 0.6161]

--- Age: Adult (n=480 images, 293 patients) ---
Metric       |     Full |     Mean |      Std |                 95% CI
-------------+----------+----------+----------+-----------------------
AUC          |   0.6092 |   0.61

## Performance on the Original Entire Test Set

Point estimates over every test image (no resampling), with the patient-level bootstrap
95% CI alongside.

In [18]:
rows = []
for model_name, full_metrics in all_full_metrics.items():
    by_group = {s['group']: s for s in all_summaries[model_name]}
    for gname, m in full_metrics.items():
        row = {'Model': model_name, 'Group': gname,
               'n_samples': m['n_samples'], 'n_patients': m['n_patients']}
        s = by_group.get(gname, {})
        for metric in METRIC_NAMES:
            row[metric] = m[metric]
            lo = s.get(f'{metric}_ci_lower', float('nan'))
            hi = s.get(f'{metric}_ci_upper', float('nan'))
            row[f'{metric}_95CI'] = f"[{lo:.4f}, {hi:.4f}]"
        rows.append(row)

full_test_df = pd.DataFrame(rows)
full_test_path = os.path.join(OUTPUT_DIR, "full_test_metrics_all_models.csv")
full_test_df.to_csv(full_test_path, index=False)
print(f"Saved: {full_test_path}\n")

print("Entire test set — Overall")
print(full_test_df[full_test_df['Group'] == 'Overall']
      [['Model', 'n_samples', 'n_patients', 'AUC', 'AUC_95CI', 'Accuracy', 'F1']]
      .to_string(index=False))

print("\nEntire test set — AUC by subgroup")
print(full_test_df.pivot(index='Group', columns='Model', values='AUC').round(4).to_string())

Saved: bootstrap_results_ml_rawage/full_test_metrics_all_models.csv

Entire test set — Overall
        Model  n_samples  n_patients      AUC         AUC_95CI  Accuracy       F1
Random Forest       1965        1224 0.627888 [0.5920, 0.6641]  0.578626 0.581818
      XGBoost       1965        1224 0.642326 [0.6079, 0.6797]  0.583206 0.594354
     CatBoost       1965        1224 0.626820 [0.5926, 0.6643]  0.576081 0.579929
          KNN       1965        1224 0.576209 [0.5385, 0.6158]  0.584224 0.675407
Decision Tree       1965        1224 0.628611 [0.5929, 0.6652]  0.581679 0.579324

Entire test set — AUC by subgroup
Model         CatBoost  Decision Tree     KNN  Random Forest  XGBoost
Group                                                                
Age: Adult      0.6043         0.5999  0.5656         0.6092   0.6166
Age: Senior     0.6054         0.6014  0.5088         0.6086   0.6122
Age: Young      0.6761         0.6682  0.6371         0.6683   0.6910
Gender: F       0.6433      

## Next step

`<Model>_predictions.csv` in `bootstrap_results_ml_rawage/` uses the same schema as
`bootstrap_evaluate.py`, so the DeLong tooling runs on it directly:

```bash
# AUC vs chance, patient-clustered
python3 statistical_testing_DeLong.py \
    --predictions bootstrap_results_ml_rawage/*_predictions.csv

# Paired against the CXR image models — same 1965 images, so fully paired
python3 compare_auc_DeLong.py \
    --set_a "bootstrap_results_ml_rawage/*_predictions.csv" \
    --set_b "bootstrap_results/exp0/MIMIC_*_predictions.csv" \
    --label_a "Demographics (raw age)" --label_b "CXR image CNN (exp0)"
```